# Yahoo Finance ('yfinance') Web Scrapping Notebook
This notebook will be the introduction to working with the yfinance api for webscrapping. We will predominantly be using this for finding ETF price and volume data but we will need to adjust it for divdends. The best thing I think we can do is divide into sectors, but also pull full index (SPY, QQQ), we will then need to find proxies for different maturity bonds (long and short) and the equivalent of a money market (1-3 month treasuries). It may be a good idea to pull currency data from this as well, possibly the DXY index of USD strength.

## Libraries

In [34]:
import numpy as np
import pandas as pd
import altair as alt  

import yfinance as yf

# Disable the max rows limit in Altair
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

ALright, let's start with just the basic pulls. I am pretty sure we should be able to pull the data using a batch pull but if not, let's pull them indvidually and add them to dataframes via a horizontal merge on date. We will only be focusing on closing prices and volume traded in the day. We can decide any data tansformations we want to use in the future.

In [35]:
# Let's create a list of sectors
sector_etfs = ["SPY","QQQ","XLF", "XLK", "XLU", "XLV", "XLE", "XLI", "XLB", "XLP", "XLY"]
prices = yf.download(sector_etfs, period='max', auto_adjust=True)[['Close','Volume']]

[*********************100%***********************]  11 of 11 completed


I'm going to separate the data pull from the data analysis so I don't have to continue to query the API as rate limiting is a known issue.

In [36]:
prices.dropna(inplace=True)
prices.head()



Price           Close                                                       \
Ticker            QQQ        SPY       XLB       XLE        XLF        XLI   
Date                                                                         
1999-03-10  43.128654  80.514305  6.032150  6.018414  12.424147  15.783933   
1999-03-11  43.339809  81.410187  6.096782  6.103727  12.484311  15.803357   
1999-03-12  42.284027  80.631157  6.070933  6.080462  12.506869  15.851918   
1999-03-15  43.498177  81.780273  6.083856  6.033928  12.544483  15.813076   
1999-03-16  43.867706  81.468666  6.058006  6.002903  12.363977  15.706226   

Price                                                  ...   Volume           \
Ticker            XLK        XLP       XLU        XLV  ...      SPY      XLB   
Date                                                   ...                     
1999-03-10  13.154647  14.296958  5.565429  19.005720  ...  3950000  53600.0   
1999-03-11  13.131363  14.437933  5.608500  19.132006  ...  6583700  27800.0   
1999-03-12  13.014946  14.562318  5.670031  19.026760  ...  5286500  15800.0   
1999-03-15  13.224494  14.686715  5.685416  19.089905  ...  5394400  13400.0   
1999-03-16  13.451497  14.753068  5.663877  19.047810  ...  4547500  15000.0   

Price                                                                  \
Ticker            XLE       XLF     XLI        XLK       XLP      XLU   
Date                                                                    
1999-03-10  3467600.0   91217.0  2700.0   884000.0   32600.0  68000.0   
1999-03-11  1818000.0  210993.0  1800.0  2432400.0   55700.0  19200.0   
1999-03-12  1208600.0  183911.0  2200.0  1672000.0   48100.0  15600.0   
1999-03-15  1046000.0   82969.0  1900.0   864200.0  246400.0  31800.0   
1999-03-16   546600.0  188466.0  4800.0  1517600.0   63700.0  20200.0   

Price                          
Ticker           XLV      XLY  
Date                           
1999-03-10   10700.0  11200.0  
1999-03-11   23600.0  26200.0  
1999-03-12   24600.0  27400.0  
1999-03-15  144800.0  14200.0  
1999-03-16    9100.0  12400.0  

[5 rows x 22 columns]

Alright, That data pull should work pretty easily let's take a look at much historical data we have. I thinmk the biggest concern is the fact that if we drop NA's we only have sector data from 2018. This does not give us a lot of exposure do different market regimes. I image this is do to the relatively new explosion of ETFs, and I imagine fixed income ETF's may contribute more to this problem. We may want to consider other options as surrogates.

In [37]:
min_date = min(prices.index)
print(f"The earliest date in our data is {min_date}.")

The earliest date in our data is 1999-03-10 00:00:00.


Ryan Peet brought up a really good idea of using mutual funds as proxies as they have been investment vehicles for a much longer period of time. So let's see if we can build the same datapull for mutual funds, and see how far back that data goes.  
Broad Market (SPY proxy): Vanguard 500 Index (VFINX) — data back to 1976, the gold standard  
Tech (XLK): Fidelity Select Technology (FSPTX) — inception 1981  
Healthcare (XLV): Fidelity Select Health Care (FSPHX) — inception 1981  
Energy (XLE): Fidelity Select Energy (FSENX) — inception 1981  
Financials (XLF): Fidelity Select Financial Services (FIDSX) — inception 1981  
Utilities (XLU): Fidelity Select Utilities (FSUTX) — inception 1981  
*Note* Industrials is really hard because it wasn't really a sector until the late 90's early '00s.
Industrials (XLI): Fidelity Select Industrials (FCYIX) — inception 1997 (this one is shorter I actually could only get data to 2019)
Industrials2 (XLI): Fidelity Select Defense & Aerospace (FSDAX)  
Consumer Staples (XLP): Fidelity Select Consumer Staples (FDFAX) — inception 1985  
Consumer Discretionary (XLY): Fidelity Select Retailing (FSRPX) as an imperfect proxy  
Materials (XLB): Fidelity Select Materials (FSDPX) — inception 1986  
Bonds (short-term): Vanguard Short-Term Bond Index (VBISX) or use direct Treasury yields from FRED  
Bonds (long-term): Vanguard Long-Term Bond Index (VBLTX) or TLT equivalent via Barclays index data from FRED  
Money market: 3-month T-bill rate from FRED is cleaner than any fund proxy  


In [38]:
# Let's create a list of sectors
sector_mfs = ["VFINX","FSPTX","FSPHX","FSENX","FIDSX","FSUTX","FSDAX","FDFAX","FSRPX","FSDPX","VBISX","VBLTX"]
prices_mfs = yf.download(sector_mfs, period='max', auto_adjust=True)[['Close','Volume']]

[*********************100%***********************]  12 of 12 completed


In [40]:
prices_mfs.dropna(inplace=True)
prices_mfs.tail()



Price           Close                                                       \
Ticker          FDFAX     FIDSX     FSDAX      FSDPX      FSENX      FSPHX   
Date                                                                         
2019-11-05  45.790173  5.098828  8.160416  45.300762  30.047283  10.517228   
2019-11-06  46.254990  5.098828  8.208673  45.019348  29.322943  10.542083   
2019-11-07  46.165184  5.135943  8.274486  45.490551  29.851286  10.591790   
2019-11-08  46.170479  5.135943  8.300808  45.641068  29.868326  10.666350   
2019-11-11  45.959187  5.131305  8.362229  45.634533  29.689373  10.641499   

Price                                                ... Volume              \
Ticker         FSPTX     FSRPX      FSUTX     VBISX  ...  FSDAX FSDPX FSENX   
Date                                                 ...                      
2019-11-05  6.591187  4.817155  58.969620  9.106738  ...    0.0   0.0   0.0   
2019-11-06  6.580577  4.808543  58.975834  9.115350  ...    0.0   0.0   0.0   
2019-11-07  6.626546  4.791319  58.627781  9.089506  ...    0.0   0.0   0.0   
2019-11-08  6.661906  4.785577  58.553207  9.089506  ...    0.0   0.0   0.0   
2019-11-11  6.676050  4.771224  58.149216  9.089506  ...    0.0   0.0   0.0   

Price                                                 
Ticker     FSPHX FSPTX FSRPX FSUTX VBISX VBLTX VFINX  
Date                                                  
2019-11-05   0.0   0.0   0.0   0.0   0.0   0.0     0  
2019-11-06   0.0   0.0   0.0   0.0   0.0   0.0     0  
2019-11-07   0.0   0.0   0.0   0.0   0.0   0.0     0  
2019-11-08   0.0   0.0   0.0   0.0   0.0   0.0     0  
2019-11-11   0.0   0.0   0.0   0.0   0.0   0.0     0  

[5 rows x 24 columns]